# LLMOps Tutorial

## Table of Contents

1. [Introduction](#introduction)
2. [Environment Setup](#setup)
3. [Prompt Engineering](#prompt-engineering)
4. [LLM APIs](#llm-apis)
5. [LangChain & LangGraph](#langchain)
6. [Structured Outputs](#structured-outputs)
7. [RAG](#rag)
8. [Evaluation](#evaluation)
9. [Production Patterns](#production-patterns)
10. [Careem Patterns](#careem-patterns)


## 1. Introduction to LLMs in Industry {#introduction}

Large Language Models have revolutionized how we build AI applications. In industry, LLMs are used for:

- **Customer Support**: Automated ticket resolution, sentiment analysis
- **Content Generation**: Marketing copy, documentation, code generation
- **Data Extraction**: Structured data from unstructured text
- **Classification**: Intent detection, content moderation
- **Reasoning**: Multi-step problem solving, decision making

### Key Challenges in Production

1. **Reliability**: LLMs can hallucinate or produce inconsistent outputs
2. **Cost**: API calls can be expensive at scale
3. **Latency**: Response times need to be acceptable for users
4. **Quality**: Outputs must meet business requirements
5. **Monitoring**: Need visibility into model performance and failures

### LLMOps: The Solution

LLMOps (LLM Operations) encompasses:
- **Prompt Management**: Version control, A/B testing, optimization
- **Evaluation**: Automated testing, human-in-the-loop feedback
- **Monitoring**: Performance metrics, error tracking, cost tracking
- **Deployment**: CI/CD pipelines, rollback strategies
- **Governance**: Compliance, safety, guardrails


## 2. Environment Setup {#setup}


In [45]:
import os
from typing import Dict, List, Optional, Any

def load_envrc():
    envrc_path = os.path.join(os.path.dirname(os.getcwd()), '.envrc')
    if not os.path.exists(envrc_path):
        envrc_path = '.envrc'
    
    if os.path.exists(envrc_path):
        env_vars = {}
        with open(envrc_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#') or line.startswith('layout'):
                    continue
                if line.startswith('export ') and '=' in line:
                    var_line = line.replace('export ', '')
                    key, value = var_line.split('=', 1)
                    value = value.strip('"').strip("'").strip()
                    if value.startswith('$'):
                        ref_key = value[1:]
                        if ref_key in env_vars:
                            value = env_vars[ref_key]
                        else:
                            continue
                    if value:
                        env_vars[key] = value
                        os.environ[key] = value

load_envrc()


In [46]:
import sys
from pydantic import field_validator
import langchain_core


In [47]:
modules_to_clear = [k for k in sys.modules.keys() if k.startswith('langchain') or k.startswith('pydantic')]
for mod in modules_to_clear:
    del sys.modules[mod]

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    openai_api_key=os.environ.get('OPENAI_API_KEY'),
)


## 3. Prompt Engineering {#prompt-engineering}


In [48]:
# Example 1: Basic Prompt
try:
    basic_prompt = """
    You are a helpful customer support agent.
    A customer says: "My order is late and I'm frustrated."
    
    Respond professionally and empathetically.
    """
    
    response = llm.invoke(basic_prompt)
    print("Basic Prompt Response:")
    print(response.content)
    print("\n" + "="*80 + "\n")
except Exception as e:
    print(f"Error: {e}")
    print("Make sure OPENAI_API_KEY is set and you have API credits")


Basic Prompt Response:
Dear [Customer's Name],

Thank you for reaching out, and I sincerely apologize for the frustration you're experiencing with your late order. I understand how important it is to receive your items on time, and I’m here to help.

Could you please provide me with your order number? I will look into the status of your order right away and see what we can do to resolve this issue for you.

Thank you for your patience, and I appreciate the opportunity to assist you.

Best regards,  
[Your Name]  
Customer Support Team




## 3. Prompt Engineering Fundamentals {#prompt-engineering}

Prompt engineering is the art and science of crafting inputs to get desired outputs from LLMs.

### Key Principles

1. **Be Specific**: Clear, unambiguous instructions
2. **Provide Context**: Include relevant background information
3. **Use Examples**: Few-shot learning improves performance
4. **Structure Output**: Request structured formats (JSON, XML, etc.)
5. **Iterate**: Test and refine prompts based on results


In [49]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",  # Using mini for cost efficiency in tutorial
    temperature=0.0,  # Lower temperature for more deterministic outputs
)

# Example 1: Basic Prompt
basic_prompt = """
You are a helpful customer support agent.
A customer says: "My order is late and I'm frustrated."

Respond professionally and empathetically.
"""

response = llm.invoke(basic_prompt)
print("Basic Prompt Response:")
print(response.content)
print("\n" + "="*80 + "\n")


Basic Prompt Response:
Dear [Customer's Name],

Thank you for reaching out, and I sincerely apologize for the frustration you're experiencing with your order. I understand how important it is to receive your items on time, and I’m here to help.

Could you please provide me with your order number? I will look into the status of your order right away and see what we can do to resolve this issue for you.

Thank you for your patience, and I appreciate the opportunity to assist you.

Best regards,  
[Your Name]  
Customer Support Team




In [50]:
# Example 2: Structured Prompt with Context
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert customer support agent for a food delivery service."),
    ("human", """
    Customer Issue: {issue}
    Order Number: {order_number}
    Customer Tier: {tier}
    
    Please provide:
    1. A brief summary of the issue
    2. Recommended action
    3. Estimated resolution time
    """)
])

messages = structured_prompt.format_messages(
    issue="Order arrived 45 minutes late and food was cold",
    order_number="ORD-12345",
    tier="Premium"
)

response = llm.invoke(messages)
print("Structured Prompt Response:")
print(response.content)
print("\n" + "="*80 + "\n")


Structured Prompt Response:
1. **Summary of the Issue**: The customer reported that their order (Order Number: ORD-12345) arrived 45 minutes late and the food was cold upon delivery.

2. **Recommended Action**: Apologize to the customer for the inconvenience caused by the late delivery and the condition of the food. Offer a refund or a credit towards their next order as compensation for the poor experience. Additionally, investigate the cause of the delay to prevent future occurrences.

3. **Estimated Resolution Time**: The customer can expect a response regarding the compensation within 24 hours, and any refund or credit will be processed within 3-5 business days.




In [51]:
# Example 3: Few-Shot Learning
examples = [
    {
        "input": "Order is late",
        "output": "I apologize for the delay. Let me check the status and provide a refund."
    },
    {
        "input": "Food quality is poor",
        "output": "I'm sorry to hear that. I'll process a full refund and escalate to our quality team."
    },
    {
        "input": "Wrong item received",
        "output": "I apologize for the mix-up. I'll arrange a replacement order immediately."
    }
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])


few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a customer support agent. Follow these examples:"),
    few_shot_prompt,
    ("human", "{input}")
])

response = final_prompt.format_messages(input="I received someone else's order")
response = llm.invoke(response)
print("Few-Shot Prompt Response:")
print(response.content)
print("\n" + "="*80 + "\n")


Few-Shot Prompt Response:
I’m sorry for the inconvenience. I’ll help you return the incorrect order and ensure you receive the correct one right away.




## 4. Working with LLM APIs {#llm-apis}

### Direct API Usage vs. LangChain

**Direct API**: More control, less abstraction
**LangChain**: Easier to use, more features, vendor-agnostic

Let's compare both approaches:


In [52]:
import openai
from langchain_openai import ChatOpenAI

# Direct OpenAI API
client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def direct_api_call(prompt: str) -> str:
    """Direct API call using OpenAI SDK"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    return response.choices[0].message.content

# LangChain wrapper
langchain_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)


def langchain_call(prompt: str) -> str:
    """LangChain wrapper call"""
    response = langchain_llm.invoke(prompt)
    return response.content

# Compare both approaches
test_prompt = "Summarize the benefits of using LangChain in 2 sentences."

print("Direct API Response:")
print(direct_api_call(test_prompt))
print("\n" + "-"*80 + "\n")
print("LangChain Response:")
print(langchain_call(test_prompt))
print("\n" + "="*80 + "\n")


Direct API Response:
LangChain streamlines the development of applications that utilize language models by providing a modular framework that integrates various components like data loaders, prompt templates, and chains. This enhances productivity and flexibility, allowing developers to create more sophisticated and efficient natural language processing solutions with ease.

--------------------------------------------------------------------------------

LangChain Response:
LangChain streamlines the development of applications that utilize language models by providing a modular framework that integrates various components like data loaders, prompt templates, and chains. This enhances productivity and flexibility, allowing developers to create more sophisticated and efficient natural language processing solutions with ease.




### Advanced API Features

- **Streaming**: Get responses as they're generated
- **Function Calling**: Structured outputs with function definitions
- **Retries**: Handle transient failures
- **Rate Limiting**: Respect API limits


In [53]:
# Streaming Example
from langchain_core.callbacks import StreamingStdOutCallbackHandler

streaming_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
    openai_api_key=os.environ.get('OPENAI_API_KEY'),
)

streaming_llm.invoke("Tell me a short story about a robot learning to code.")
print("\n" + "="*80 + "\n")


In a bustling city of the future, where skyscrapers gleamed with digital screens and drones zipped through the air, there lived a curious little robot named Byte. Byte was a maintenance bot, designed to keep the city’s infrastructure running smoothly. However, Byte had a secret: it longed to create, to build something of its own.

One day, while cleaning the local tech library, Byte stumbled upon a dusty old book titled "Introduction to Coding." Its circuits buzzed with excitement as it scanned the pages filled with colorful code snippets and diagrams. Byte had never seen anything like it. The idea of writing instructions to make machines do things fascinated it.

Determined to learn, Byte began to study the book during its downtime. It practiced simple commands, like making lights blink and sounds play. Each successful line of code filled Byte with joy, and soon it was ready for a bigger challenge: creating a small game.

Byte decided to make a game where players could help a lost rob

In [54]:
# Retry and Error Handling
from langchain_core.runnables import RunnableLambda
from tenacity import retry, stop_after_attempt, wait_exponential

llm_with_retry = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    max_retries=3,
    timeout=30,
    openai_api_key=os.environ.get('OPENAI_API_KEY'),
)

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def robust_llm_call(prompt: str):
    """LLM call with custom retry logic"""
    return llm_with_retry.invoke(prompt).content

result = robust_llm_call("What is 2+2?")
print(result)
print("\n" + "="*80 + "\n")


2 + 2 equals 4.




## 5. LangChain & LangGraph {#langchain}

LangChain provides building blocks for LLM applications. LangGraph extends this with stateful, multi-step workflows.

### Core Concepts

- **Runnables**: Composable units that can be chained together
- **Chains**: Sequences of operations
- **Agents**: LLMs that can use tools
- **Graphs**: Stateful workflows with conditional logic


In [55]:
# Example: Simple Chain
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate

# Step 1: Extract key information
extract_chain = ChatPromptTemplate.from_template(
    "Extract the main topic from: {input}"
) | llm | RunnableLambda(lambda x: x.content)

# Step 2: Generate response
response_chain = ChatPromptTemplate.from_template(
    "Write a brief explanation about: {topic}"
) | llm | RunnableLambda(lambda x: x.content) |

# Combine chains
full_chain = RunnablePassthrough.assign(topic=extract_chain) | response_chain

result = full_chain.invoke({"input": "quantum computing"})
print("Chain Result:")
print(result)
print("\n" + "="*80 + "\n")


╭──────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ) | llm | RunnableLambda(lambda x: x.content) |                                                  │
│                                                ▲                                                 │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
SyntaxError: invalid syntax

## 6. Structured Outputs & Guardrails {#structured-outputs}

Production LLM applications need reliable, structured outputs. Pydantic models help enforce schemas.


In [ ]:
from pydantic import BaseModel, Field
from typing import List

class TicketAnalysis(BaseModel):
    sentiment: str = Field(description="Overall sentiment: positive, neutral, or negative")
    urgency: str = Field(description="Urgency level: low, medium, or high")
    category: str = Field(description="Ticket category")
    key_issues: List[str] = Field(description="List of key issues mentioned")
    suggested_action: str = Field(description="Recommended action for support agent")
    confidence_score: float = Field(description="Confidence score between 0 and 1", ge=0, le=1)

ticket_text = """
I ordered food 2 hours ago and it still hasn't arrived. 
This is the third time this week. I want a refund and compensation.
"""

prompt = ChatPromptTemplate.from_template("Analyze this customer ticket: {ticket}")
chain = prompt | llm.with_structured_output(schema=TicketAnalysis)

result = chain.invoke({"ticket": ticket_text})

print(f"Sentiment: {result.sentiment}")
print(f"Urgency: {result.urgency}")
print(f"Category: {result.category}")
print(f"Key Issues: {result.key_issues}")
print(f"Suggested Action: {result.suggested_action}")
print(f"Confidence: {result.confidence_score}")


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:20                                                                                   │
│                                                                                                  │
│   17 prompt = ChatPromptTemplate.from_template("Analyze this customer ticket: {ticket}")         │
│   18 chain = prompt | llm.with_structured_output(schema=TicketAnalysis)                          │
│   19                                                                                             │
│ ❱ 20 result = chain.invoke({"ticket": ticket_text})                                              │
│   21                                                                                             │
│   22 print(f"Sentiment: {result.sentiment}")                                                     │
│   23 print(f"Urgency: {result.urgency}")                                                         │
│                                                                                                  │
│ /Users/sualeh.ali/work/ai-care/.venv/lib/python3.12/site-packages/langchain_core/runnables/base. │
│ py:3046 in invoke                                                                                │
│                                                                                                  │
│   3043 │   │   │   │   │   if i == 0:                                                            │
│   3044 │   │   │   │   │   │   input_ = context.run(step.invoke, input_, config, **kwargs)       │
│   3045 │   │   │   │   │   else:                                                                 │
│ ❱ 3046 │   │   │   │   │   │   input_ = context.run(step.invoke, input_, config)                 │
│   3047 │   │   # finish the root run                                                             │
│   3048 │   │   except BaseException as e:                                                        │
│   3049 │   │   │   run_manager.on_chain_error(e)                                                 │
│                                                                                                  │
│ /Users/sualeh.ali/work/ai-care/.venv/lib/python3.12/site-packages/langchain_core/runnables/base. │
│ py:5434 in invoke                                                                                │
│                                                                                                  │
│   5431 │   │   config: Optional[RunnableConfig] = None,                                          │
│   5432 │   │   **kwargs: Optional[Any],                                                          │
│   5433 │   ) -> Output:                                                                          │
│ ❱ 5434 │   │   return self.bound.invoke(                                                         │
│   5435 │   │   │   input,                                                                        │
│   5436 │   │   │   self._merge_configs(config),                                                  │
│   5437 │   │   │   **{**self.kwargs, **kwargs},                                                  │
│                                                                                                  │
│ /Users/sualeh.ali/work/ai-care/.venv/lib/python3.12/site-packages/langchain_core/language_models │
│ /chat_models.py:395 in invoke                                                                    │
│                                                                                                  │
│    392 │   │   config = ensure_config(config)                                                    │
│    393 │   │   return cast(                                                                      │
│    394 │   │   │   "ChatGeneration",                                                             │
│ ❱  395 │   │   │   self.generate_prompt(                   

In [ ]:
# Guardrails: Validate outputs before proceeding
from langchain_core.runnables import RunnableLambda
from langchain_core.exceptions import OutputParserException

class RefundDecision(BaseModel):
    """Decision on refund eligibility"""
    eligible: bool
    amount: float = Field(ge=0, description="Refund amount in currency")
    reason: str

def validate_refund(decision: RefundDecision) -> RefundDecision:
    """Guardrail: Ensure refund amount is reasonable"""
    if decision.eligible and decision.amount > 1000:
        raise ValueError(f"Refund amount {decision.amount} exceeds maximum allowed (1000)")
    return decision

refund_llm = llm.with_structured_output(RefundDecision)

# Chain with validation
refund_chain = (
    ChatPromptTemplate.from_template(
        "Determine if a refund is eligible for this ticket: {ticket}"
    )
    | refund_llm
    | RunnableLambda(validate_refund)
)

# Test with fallback handling
from langchain_core.runnables import RunnableWithFallbacks

refund_chain_with_fallback = refund_chain.with_fallbacks([
    RunnableLambda(lambda x: RefundDecision(eligible=False, amount=0.0, reason="Error processing request"))
])

try:
    result = refund_chain_with_fallback.invoke({
        "ticket": "Food was cold and arrived late. Order total was $50."
    })
    print("Refund Decision:")
    print(f"Eligible: {result.eligible}")
    print(f"Amount: ${result.amount}")
    print(f"Reason: {result.reason}")
except Exception as e:
    print(f"Error: {e}")
print("\n" + "="*80 + "\n")


Refund Decision:
Eligible: True
Amount: $25.0
Reason: Food was cold and arrived late, which significantly impacted the dining experience.




## 7. RAG {#rag}

RAG combines LLMs with external knowledge bases for context-aware responses.


In [ ]:
try:
    from langchain_community.vectorstores import FAISS
    from langchain_openai import OpenAIEmbeddings
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    from langchain_core.documents import Document
    
    knowledge_base = [
        "Refund policy: Full refunds for orders over 30 minutes late.",
        "Premium members get priority support.",
        "Food quality complaints handled within 24 hours.",
    ] 
    
    embeddings = OpenAIEmbeddings(openai_api_key=os.environ.get('OPENAI_API_KEY'))
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
    docs = [Document(page_content=text) for text in knowledge_base]
    splits = text_splitter.split_documents(docs)
    vectorstore = FAISS.from_documents(splits, embeddings)
    
    retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
    
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
    
    rag_prompt = ChatPromptTemplate.from_template("Answer based on context:\n{context}\n\nQuestion: {question}")
    rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} | rag_prompt | llm)
    
    response = rag_chain.invoke("What is the refund policy for late deliveries?")
    print(response.content)
except ImportError:
    knowledge_base = "Refund policy: Full refunds for orders over 30 minutes late."
    response = llm.invoke(f"Context: {knowledge_base}\n\nQuestion: What is the refund policy?")
    print(response.content)
print("\n" + "="*80 + "\n")


The refund policy states that customers are eligible for a full refund if their orders are more than 30 minutes late.




In [ ]:
# LangSmith Tracing
from langchain_core.tracers import LangChainTracer
from langsmith import Client

# Enable tracing (set LANGCHAIN_TRACING_V2=true and LANGCHAIN_API_KEY)
# tracer = LangChainTracer(project_name="llmops-tutorial")

# Example: Traced chain
traced_chain = (
    ChatPromptTemplate.from_template("Summarize: {text}")
    | llm
)

# Run with tracing (uncomment when LangSmith is configured)
# result = traced_chain.invoke({"text": "Customer support is important for business success."})
# print("Traced execution - check LangSmith dashboard")

print("LangSmith tracing configured (requires API key)")
print("\n" + "="*80 + "\n")


LangSmith tracing configured (requires API key)




In [ ]:
try:
    from langchain.evaluation import Criteria, EvaluatorType, load_evaluator
    
    evaluator = load_evaluator(EvaluatorType.CRITERIA, criteria=Criteria.HELPFULNESS, llm=llm)
    result = evaluator.evaluate_strings(
        prediction="We offer full refunds for orders over 30 minutes late.",
        input="What is your refund policy?"
    )
    print(f"Score: {result['score']}")
except ImportError:
    print("Install: pip install langchain[evaluation]")


Score: 0


## 10. Careem Patterns {#careem-patterns}

Production patterns: multi-step pipelines, dynamic schemas, guardrails, policy evaluation, async processing.


In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

class TicketContext(BaseModel):
    """Immutable context object (pattern from Careem)"""
    ticket_id: str
    customer_message: str
    is_ticket_supported: bool = True
    disposition: Optional[str] = None
    guardrails_passed: bool = True
    refund_amount: Optional[float] = None
    response: Optional[str] = None

async def process_ticket_pipeline(context: TicketContext) -> TicketContext:
    """
    Multi-step pipeline with early returns (Careem pattern)
    Each step validates and can short-circuit the pipeline
    """
    # Step 1: Validate ticket is processable
    if not context.customer_message or len(context.customer_message) < 10:
        context = context.model_copy(update={"is_ticket_supported": False})
        return context
    
    # Step 2: Classify disposition
    class DispositionResult(BaseModel):
        disposition: str = Field(description="Ticket disposition code")
        confidence: float = Field(ge=0, le=1)
        is_valid: bool = Field(description="Whether ticket is valid")
    
    disposition_llm = llm.with_structured_output(DispositionResult)
    disposition = await disposition_llm.ainvoke(
        f"Classify this ticket: {context.customer_message}"
    )
    
    context = context.model_copy(update={
        "disposition": disposition.disposition,
        "is_ticket_supported": disposition.is_valid
    })
    
    if not context.is_ticket_supported:
        return context  # Early return
    
    # Step 3: Guardrails check
    if not context.guardrails_passed:
        return context  # Early return
    
    # Step 4: Calculate refund
    if disposition.disposition == "refund_eligible":
        context = context.model_copy(update={"refund_amount": 50.0})
    
    # Step 5: Generate response
    response_prompt = f"Generate response for: {context.customer_message}"
    response = await llm.ainvoke(response_prompt)
    context = context.model_copy(update={"response": response.content})
    
    return context

# Demo
ticket = TicketContext(
    ticket_id="TKT-001",
    customer_message="My order arrived 45 minutes late and food was cold"
)

result = await process_ticket_pipeline(ticket)
print(f"Disposition: {result.disposition}")
print(f"Supported: {result.is_ticket_supported}")
print(f"Refund: ${result.refund_amount if result.refund_amount else 0}")
print(f"Response: {result.response[:100] if result.response else 'N/A'}...")
print("\n" + "="*80 + "\n")


Disposition: Late Delivery
Supported: True
Refund: $0
Response: I'm really sorry to hear that your order arrived late and cold. That’s not the experience we want fo...




In [ ]:
from dataclasses import dataclass
from typing import Sequence, Type, Dict, Tuple
from pydantic import create_model

@dataclass(frozen=True)
class RuleSpec:
    """Rule specification for dynamic guardrails"""
    field_name: str
    description: str

class GuardrailCheck(BaseModel):
    """Base structure for each guardrail"""
    reason: str
    is_valid: bool = True

def build_guardrails_model(rule_specs: Sequence[RuleSpec]) -> Type[BaseModel]:
    """
    Dynamically build a Pydantic model from rule specs (Careem pattern)
    Creates a model with one GuardrailCheck field per rule + aggregate fields
    """
    # Per-rule fields
    fields: Dict[str, Tuple[type, Field]] = {
        spec.field_name: (
            GuardrailCheck,
            Field(..., description=spec.description)
        )
        for spec in rule_specs
    }
    
    # Aggregate fields
    fields.update({
        "all_guardrails_followed": (
            bool,
            Field(True, description="True if all guardrails passed")
        ),
        "decision_summary": (
            str,
            Field(..., description="Summary of guardrails evaluation")
        ),
        "actor": (
            str,
            Field("", description="Reasons why guardrails might be correct")
        ),
        "critic": (
            str,
            Field("", description="Reasons why guardrails might be incorrect")
        ),
    })
    
    return create_model("Guardrails", **fields)

# Define rules
rules = [
    RuleSpec(
        field_name="order_item_match_check",
        description="Verify ordered items match the complaint"
    ),
    RuleSpec(
        field_name="not_condiment_check",
        description="Ensure complaint is not only about condiments"
    ),
]

# Build model dynamically
GuardrailsModel = build_guardrails_model(rules)

# Use with LLM
guardrails_llm = llm.with_structured_output(GuardrailsModel)

ticket_data = """
Customer complaint: "My burger was missing"
Order: Burger, Fries, Drink
"""

result = await guardrails_llm.ainvoke(
    f"Evaluate guardrails for: {ticket_data}"
)

print("Guardrails Evaluation:")
print(f"Order Match: {result.order_item_match_check.is_valid}")
print(f"Not Condiment: {result.not_condiment_check.is_valid}")
print(f"All Passed: {result.all_guardrails_followed}")
print(f"Summary: {result.decision_summary}")
print("\n" + "="*80 + "\n")


Guardrails Evaluation:
Order Match: True
Not Condiment: True
All Passed: True
Summary: Both guardrails have been satisfied: the complaint matches the order and is not solely about condiments.




In [ ]:
from typing import Literal

def build_disposition_model(disposition_codes: list[str]) -> Type[BaseModel]:
    """Build disposition classification model with ranked list"""
    
    # Create Literal type for valid codes
    disposition_literal = Literal.__getitem__(tuple(disposition_codes))
    
    # Inner model for each candidate
    ClassifiedInfo = create_model(
        "ClassifiedInfo",
        disposition=(disposition_literal, Field(..., description="Disposition code")),
        reason=(str, Field(..., description="Rationale for this classification")),
        confidence=(float, Field(ge=0, le=1, description="Confidence score")),
    )
    
    # Top-level model
    return create_model(
        "DispositionClassification",
        is_valid_request=(bool, Field(..., description="Is ticket valid")),
        classified_disposition_list=(
            list[ClassifiedInfo],
            Field(..., description="Ranked list of disposition candidates")
        ),
    )

# Define disposition codes
disposition_codes = [
    "late_delivery",
    "wrong_item",
    "food_quality",
    "missing_item",
    "not_supported"
]

DispositionModel = build_disposition_model(disposition_codes)
disposition_llm = llm.with_structured_output(DispositionModel)

ticket = "My order arrived 45 minutes late and the food was cold"

result = await disposition_llm.ainvoke(f"Classify: {ticket}")

print("Disposition Classification:")
print(f"Valid Request: {result.is_valid_request}")
print(f"\nRanked Candidates:")
for i, candidate in enumerate(result.classified_disposition_list[:3], 1):
    print(f"{i}. {candidate.disposition} (confidence: {candidate.confidence:.2f})")
    print(f"   Reason: {candidate.reason}")
print("\n" + "="*80 + "\n")


Disposition Classification:
Valid Request: True

Ranked Candidates:
1. late_delivery (confidence: 0.85)
   Reason: The order arrived 45 minutes later than expected, indicating a delay in delivery.
2. food_quality (confidence: 0.75)
   Reason: The food was cold upon arrival, which affects the quality of the order.




In [ ]:
import pandas as pd
from enum import Enum

class CriterionValue(str, Enum):
    TRUE = "TRUE"
    FALSE = "FALSE"
    OPTIONAL = "OPTIONAL"
    NONE = "NONE"

class PolicyEvaluator:
    """Evaluate policies based on criteria (simplified Careem pattern)"""
    
    def __init__(self, policy_df: pd.DataFrame):
        self.policy_df = policy_df
    
    def evaluate(
        self,
        status: str,
        ticket_age_hours: float,
        has_attachment: bool
    ) -> Dict[str, any]:
        """Evaluate policy against criteria"""
        
        # Filter by status
        matching_rows = self.policy_df[
            self.policy_df['status'].str.upper() == status.upper()
        ]
        
        if matching_rows.empty:
            return {"matched": False, "actions": []}
        
        # Evaluate criteria for each row
        for _, row in matching_rows.iterrows():
            criteria_met = True
            
            # Check ticket age criteria
            if pd.notna(row.get('ticket_age_hours')):
                expected_age = float(row['ticket_age_hours'])
                if abs(ticket_age_hours - expected_age) > 1.0:
                    criteria_met = False
                    continue
            
            # Check attachment criteria
            if pd.notna(row.get('requires_attachment')):
                if row['requires_attachment'].upper() == "TRUE" and not has_attachment:
                    criteria_met = False
                    continue
            
            if criteria_met:
                actions = []
                if row.get('send_email') == "TRUE":
                    actions.append("send_email")
                if row.get('update_ticket') == "TRUE":
                    actions.append("update_ticket")
                
                return {
                    "matched": True,
                    "actions": actions,
                    "disposition": row.get('disposition_code', '')
                }
        
        return {"matched": False, "actions": []}

# Create sample policy DataFrame
policy_data = {
    'status': ['COMPLETED', 'COMPLETED', 'IN_PROGRESS'],
    'ticket_age_hours': [2.0, 24.0, 12.0],
    'requires_attachment': ['FALSE', 'TRUE', 'FALSE'],
    'send_email': ['TRUE', 'TRUE', 'FALSE'],
    'update_ticket': ['FALSE', 'TRUE', 'TRUE'],
    'disposition_code': ['REFUND', 'INVESTIGATE', 'WAIT']
}

policy_df = pd.DataFrame(policy_data)
evaluator = PolicyEvaluator(policy_df)

# Evaluate
result = evaluator.evaluate(
    status="COMPLETED",
    ticket_age_hours=2.5,
    has_attachment=False
)

print("Policy Evaluation:")
print(f"Matched: {result['matched']}")
print(f"Actions: {result['actions']}")
print(f"Disposition: {result.get('disposition', 'N/A')}")
print("\n" + "="*80 + "\n")


Policy Evaluation:
Matched: True
Actions: ['send_email']
Disposition: REFUND




In [ ]:
from langchain_core.runnables import Runnable

class PromptManager:
    """Simplified version of Careem's LangSmith prompt management"""
    
    def __init__(self):
        self._cache = {}
    
    def get_prompt(self, prompt_name: str) -> Runnable:
        """Get prompt from cache or create new one"""
        if prompt_name not in self._cache:
            # In production, this would fetch from LangSmith Hub
            # For demo, we'll use templates
            prompt_map = {
                "disposition_classification": ChatPromptTemplate.from_template(
                    "Classify this customer ticket into one of: late_delivery, wrong_item, food_quality, missing_item\n\nTicket: {ticket}"
                ),
                "response_generation": ChatPromptTemplate.from_template(
                    "Generate a professional customer support response for:\nTicket: {ticket}\nDisposition: {disposition}\nRefund: ${refund}"
                ),
                "guardrails_check": ChatPromptTemplate.from_template(
                    "Evaluate guardrails for this ticket:\n{ticket}\n\nCheck: {check_description}"
                ),
            }
            
            if prompt_name in prompt_map:
                self._cache[prompt_name] = prompt_map[prompt_name]
            else:
                # Fallback
                self._cache[prompt_name] = ChatPromptTemplate.from_template("{ticket}")
        
        return self._cache[prompt_name]

# Usage
prompt_manager = PromptManager()

# Get prompts (cached after first access)
disposition_prompt = prompt_manager.get_prompt("disposition_classification")
response_prompt = prompt_manager.get_prompt("response_generation")

# Use in chain
disposition_chain = disposition_prompt | llm
response_chain = response_prompt | llm

ticket_text = "Order arrived late"

# Classify
disposition_result = await disposition_chain.ainvoke({"ticket": ticket_text})
print(f"Disposition: {disposition_result.content[:50]}...")

# Generate response
response_result = await response_chain.ainvoke({
    "ticket": ticket_text,
    "disposition": "late_delivery",
    "refund": 50
})
print(f"Response: {response_result.content[:100]}...")
print("\n" + "="*80 + "\n")


Disposition: Classification: late_delivery...
Response: Subject: Your Order Delivery Update

Dear [Customer's Name],

Thank you for reaching out to us regar...




In [ ]:

import asyncio
from dataclasses import dataclass
from typing import List, Dict, Any
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate

@dataclass
class ActionSpec:
    name: str
    prompt_id: str
    type: str

class ActionResponse(BaseModel):
    action: str
    message: str
    type: str

async def generate_action_responses(
    actions: List[ActionSpec],
    context: Dict[str, Any]
) -> List[ActionResponse]:
    async def generate_one(action: ActionSpec) -> ActionResponse:
        prompt_template = ChatPromptTemplate.from_template(
            f"Generate {action.name} response: {context.get('ticket', '')}"
        )
        chain = prompt_template | llm
        result = await chain.ainvoke(context)
        
        return ActionResponse(
            action=action.name,
            message=result.content,
            type=action.type
        )
    
    # Process all actions concurrently
    results = await asyncio.gather(*[generate_one(a) for a in actions])
    return list(results)

# Demo
actions = [
    ActionSpec(name="send_email", prompt_id="email_prompt", type="email"),
    ActionSpec(name="update_ticket", prompt_id="ticket_prompt", type="ticket"),
]

context = {
    "ticket": "Order was late",
    "disposition": "late_delivery",
    "refund": 50.0
}

responses = await generate_action_responses(actions, context)

print("Action Responses (generated in parallel):")
for resp in responses:
    print(f"\n{resp.action} ({resp.type}):")
    print(f"  {resp.message[:80]}...")
print("\n" + "="*80 + "\n")


Action Responses (generated in parallel):

send_email (email):
  Subject: Update on Your Order Status

Dear [Customer's Name],

Thank you for rea...

update_ticket (ticket):
  Subject: Update on Your Ticket - Order Delay

Dear [Customer's Name],

Thank you...




In [ ]:


from time import time_ns
from typing import Dict
from collections import defaultdict

class SkillMetrics:
    """Track metrics per skill (simplified Careem pattern)"""
    
    def __init__(self):
        self.metrics: Dict[str, Dict] = defaultdict(lambda: {
            "count": 0,
            "total_time_ns": 0,
            "errors": 0
        })
    
    def record(self, skill_name: str, duration_ns: int, success: bool = True):
        """Record a metric"""
        self.metrics[skill_name]["count"] += 1
        self.metrics[skill_name]["total_time_ns"] += duration_ns
        if not success:
            self.metrics[skill_name]["errors"] += 1
    
    def get_stats(self, skill_name: str) -> Dict:
        """Get statistics for a skill"""
        m = self.metrics[skill_name]
        if m["count"] == 0:
            return {}
        return {
            "count": m["count"],
            "avg_time_ms": (m["total_time_ns"] / m["count"]) / 1_000_000,
            "error_rate": m["errors"] / m["count"],
            "total_time_ms": m["total_time_ns"] / 1_000_000
        }

metrics = SkillMetrics()

# Track a skill execution
async def tracked_skill_execution(skill_name: str, operation):
    """Execute operation with metrics tracking"""
    start_ns = time_ns()
    try:
        result = await operation()
        duration_ns = time_ns() - start_ns
        metrics.record(skill_name, duration_ns, success=True)
        
        return result
    except Exception as e:
        duration_ns = time_ns() - start_ns
        metrics.record(skill_name, duration_ns, success=False)
        raise

# Demo
result1 = await tracked_skill_execution(
    "disposition_classification",
    lambda: llm.ainvoke("Classify: Order is late")
)

result2 = await tracked_skill_execution(
    "response_generation",
    lambda: llm.ainvoke("Generate response: Order is late")
)

print("Metrics:")
for skill in ["disposition_classification", "response_generation"]:
    stats = metrics.get_stats(skill)
    print(f"{skill}:")
    print(f"  Count: {stats['count']}")
    print(f"  Avg Time: {stats['avg_time_ms']:.2f}ms")
    print(f"  Total Time: {stats['total_time_ms']:.2f}ms")
print("\n" + "="*80 + "\n")


Metrics:
disposition_classification:
  Count: 1
  Avg Time: 1539.49ms
  Total Time: 1539.49ms
response_generation:
  Count: 1
  Avg Time: 2461.35ms
  Total Time: 2461.35ms




### Key Insights from Careem's Implementation

1. **Immutable Context Pattern**: Use `model_copy()` to update state, ensuring thread-safety
2. **Early Returns**: Validate at each step and short-circuit if validation fails
3. **Dynamic Models**: Build Pydantic schemas at runtime for flexibility
4. **Fallback Chains**: Always have fallbacks for critical operations
5. **Prompt Caching**: Cache prompts from LangSmith Hub to reduce API calls
6. **Structured Outputs**: Use Pydantic models for all LLM outputs
7. **Metrics Per Skill**: Track performance at granular level
8. **Policy-Driven Logic**: Separate business rules from code using CSV/config
9. **Actor/Critic Pattern**: Self-reflection improves guardrail accuracy
10. **Parallel Processing**: Use async/await for independent operations

### Production Checklist (Based on Careem)

- ✅ Multi-step pipeline with validation gates
- ✅ Dynamic schema building for flexibility
- ✅ Guardrails with self-reflection
- ✅ Prompt management via LangSmith Hub
- ✅ Metrics tracking per operation
- ✅ Fallback handling for all LLM calls
- ✅ Immutable context objects
- ✅ Policy-based evaluation
- ✅ Async processing for performance
- ✅ Structured outputs everywhere


## 10. Deployment & Scaling {#deployment}


### Complete Example: End-to-End Ticket Processing (Careem Style)

Combining all patterns into a production-ready pipeline:


In [ ]:
# Complete Production Pipeline (Combining All Careem Patterns)

class ProductionTicketPipeline:
    """Complete ticket processing pipeline using Careem patterns"""
    
    def __init__(self, llm: ChatOpenAI):
        self.llm = llm
        self.metrics = SkillMetrics()
        self.prompt_manager = PromptManager()
        
        # Build dynamic models
        self.guardrails_model = build_guardrails_model([
            RuleSpec("order_match", "Verify order matches complaint"),
            RuleSpec("not_condiment", "Ensure not condiment-only issue"),
        ])
        
        self.disposition_model = build_disposition_model([
            "late_delivery", "wrong_item", "food_quality", "missing_item", "not_supported"
        ])
    
    async def process(self, ticket: TicketContext) -> TicketContext:
        """Process ticket through complete pipeline"""
        
        # Step 1: Disposition Classification
        async def classify():
            prompt = self.prompt_manager.get_prompt("disposition_classification")
            chain = prompt | self.llm.with_structured_output(self.disposition_model)
            return await chain.ainvoke({"ticket": ticket.customer_message})
        
        disposition_result = await tracked_skill_execution("disposition", classify)
        
        if not disposition_result.is_valid_request:
            return ticket.model_copy(update={"is_ticket_supported": False})
        
        ticket = ticket.model_copy(update={
            "disposition": disposition_result.classified_disposition_list[0].disposition
        })
        
        # Step 2: Guardrails Check
        async def check_guardrails():
            guardrails_llm = self.llm.with_structured_output(self.guardrails_model)
            return await guardrails_llm.ainvoke(
                f"Ticket: {ticket.customer_message}\nDisposition: {ticket.disposition}"
            )
        
        guardrails_result = await tracked_skill_execution("guardrails", check_guardrails)
        
        if not guardrails_result.all_guardrails_followed:
            return ticket.model_copy(update={"guardrails_passed": False})
        
        # Step 3: Generate Response
        async def generate_response():
            prompt = self.prompt_manager.get_prompt("response_generation")
            chain = prompt | self.llm
            return await chain.ainvoke({
                "ticket": ticket.customer_message,
                "disposition": ticket.disposition,
                "refund": ticket.refund_amount or 0
            })
        
        response_result = await tracked_skill_execution("response", generate_response)
        
        return ticket.model_copy(update={"response": response_result.content})

# Demo
pipeline = ProductionTicketPipeline(llm)

ticket = TicketContext(
    ticket_id="TKT-001",
    customer_message="My order arrived 45 minutes late and the food was cold. I want a refund."
)

result = await pipeline.process(ticket)

print("Complete Pipeline Result:")
print(f"Supported: {result.is_ticket_supported}")
print(f"Disposition: {result.disposition}")
print(f"Guardrails Passed: {result.guardrails_passed}")
print(f"\nResponse:\n{result.response[:200] if result.response else 'N/A'}...")

print("\n\nPipeline Metrics:")
for skill in ["disposition", "guardrails", "response"]:
    stats = metrics.get_stats(skill)
    if stats:
        print(f"{skill}: {stats['avg_time_ms']:.0f}ms avg")
print("\n" + "="*80 + "\n")


Complete Pipeline Result:
Supported: True
Disposition: late_delivery
Guardrails Passed: True

Response:
Subject: Response to Your Order Concern

Dear [Customer's Name],

Thank you for reaching out to us regarding your recent order. We sincerely apologize for the inconvenience caused by the late delivery...


Pipeline Metrics:
disposition: 2363ms avg
guardrails: 2912ms avg
response: 5383ms avg




In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",  # Using mini for cost efficiency in tutorial
    temperature=0.0,  # Lower temperature for more deterministic outputs
)

# Example 1: Basic Prompt
basic_prompt = """
You are a helpful customer support agent.
A customer says: "My order is late and I'm frustrated."

Respond professionally and empathetically.
"""

response = llm.invoke(basic_prompt)
print("Basic Prompt Response:")
print(response.content)
print("\n" + "="*80 + "\n")


Basic Prompt Response:
Dear [Customer's Name],

Thank you for reaching out, and I sincerely apologize for the frustration you're experiencing with your order. I understand how important it is to receive your items on time, and I’m here to help.

Could you please provide me with your order number? I will look into the status of your order and do my best to resolve this issue for you as quickly as possible.

Thank you for your patience, and I appreciate your understanding.

Best regards,  
[Your Name]  
Customer Support Team




In [ ]:
# Argilla: Human-in-the-Loop Evaluation
# Note: Requires Argilla server setup
try:
    import argilla as rg
    
    # Uncomment when Argilla is configured
    # rg.init(
    #     api_url="http://localhost:6900",
    #     api_key="your-api-key"
    # )
    
    # Create evaluation dataset
    def create_evaluation_dataset():
        """Create a dataset for human evaluation"""
        records = []
        
        test_inputs = [
            "My order is late",
            "Food quality was poor",
            "I want a refund"
        ]
        
        for input_text in test_inputs:
            # Generate response
            response = llm.invoke(f"Respond to customer: {input_text}").content
            
            # Create Argilla record (commented out - requires server)
            # record = rg.TextClassificationRecord(
            #     text=input_text,
            #     annotation="",  # To be filled by human annotators
            #     metadata={"generated_response": response}
            # )
            # records.append(record)
            records.append({"input": input_text, "response": response})
        
        return records
    
    dataset = create_evaluation_dataset()
    print("Argilla evaluation dataset creation (requires server setup)")
    print(f"Created {len(dataset)} records")
except ImportError:
    print("Argilla not installed. Install with: pip install argilla")
print("\n" + "="*80 + "\n")


Argilla evaluation dataset creation (requires server setup)
Created 3 records




## 9. Production Patterns {#production-patterns}

### Key Patterns for Production LLM Applications

1. **Caching**: Reduce costs and latency
2. **Rate Limiting**: Respect API limits
3. **Circuit Breakers**: Handle failures gracefully
4. **Fallbacks**: Alternative models/strategies
5. **Batching**: Process multiple requests efficiently
6. **Async Processing**: Handle concurrent requests


In [ ]:
# Caching: Reduce API calls and costs
from langchain.cache import InMemoryCache
from langchain.globals import set_llm_cache
import time

# Enable caching
set_llm_cache(InMemoryCache())

# Test caching
prompt = "Explain quantum computing in simple terms"

print("First call (will hit API):")
start = time.time()
result1 = llm.invoke(prompt)
time1 = time.time() - start
print(f"Time: {time1:.2f}s")
print(f"Response length: {len(result1.content)} chars\n")

print("Second call (cached):")
start = time.time()
result2 = llm.invoke(prompt)
time2 = time.time() - start
print(f"Time: {time2:.2f}s")
print(f"Response length: {len(result2.content)} chars")
print(f"Speedup: {time1/time2:.1f}x faster")
print("\n" + "="*80 + "\n")


First call (will hit API):
Time: 7.70s
Response length: 1784 chars

Second call (cached):
Time: 0.01s
Response length: 1784 chars
Speedup: 1305.5x faster




In [ ]:
# Fallbacks: Use alternative models when primary fails
from langchain_core.runnables import RunnableWithFallbacks
from langchain_openai import ChatOpenAI

# Primary model
primary_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# Fallback model (cheaper/faster alternative)
fallback_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)

# Chain with fallback
robust_chain = (
    ChatPromptTemplate.from_template("Answer: {question}")
    | primary_llm
).with_fallbacks([fallback_llm])

result = robust_chain.invoke({"question": "What is machine learning?"})
print("Fallback Chain Result:")
print(result.content)
print("\n" + "="*80 + "\n")


Fallback Chain Result:
Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit programming. Instead of being programmed with specific instructions for every task, machine learning systems learn from data, identifying patterns and making decisions based on that data.

There are several key components and concepts in machine learning:

1. **Data**: Machine learning relies on large amounts of data to train models. This data can be structured (like databases) or unstructured (like images or text).

2. **Algorithms**: These are the mathematical models and techniques used to analyze data and make predictions. Common algorithms include decision trees, neural networks, support vector machines, and clustering methods.

3. **Training**: The process of feeding data into a machine learning model so it can learn from it. During training, the model adjusts its parameters 

In [ ]:
# Async Processing: Handle multiple requests concurrently
import asyncio
from langchain_openai import ChatOpenAI

async def process_tickets_async(tickets: List[str]) -> List[str]:
    """Process multiple tickets concurrently"""
    async_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    
    async def process_one(ticket: str) -> str:
        prompt = f"Generate a response for: {ticket}"
        response = await async_llm.ainvoke(prompt)
        return response.content
    
    # Process all tickets concurrently
    results = await asyncio.gather(*[process_one(ticket) for ticket in tickets])
    return results

# Test async processing
test_tickets = [
    "Order is late",
    "Food quality issue",
    "Payment problem"
]

import time

print("Processing tickets asynchronously...")
start = time.time()
results = await process_tickets_async(test_tickets)
async_time = time.time() - start

print(f"Processed {len(test_tickets)} tickets in {async_time:.2f}s")
for i, (ticket, response) in enumerate(zip(test_tickets, results), 1):
    print(f"\n{i}. Ticket: {ticket}")
    print(f"   Response: {response[:50]}...")
print("\n" + "="*80 + "\n")


Processing tickets asynchronously...
Processed 3 tickets in 5.00s

1. Ticket: Order is late
   Response: I apologize for the inconvenience regarding your l...

2. Ticket: Food quality issue
   Response: Subject: Concern Regarding Food Quality

Dear [Rec...

3. Ticket: Payment problem
   Response: I'm sorry to hear that you're experiencing a payme...




In [ ]:
# Batching: Process multiple inputs efficiently
from langchain_core.batch import run_batch

prompts = [
    "Summarize: Machine learning is a subset of AI",
    "Summarize: Deep learning uses neural networks",
    "Summarize: NLP processes human language"
]

print("Batch processing:")
start = time.time()
batch_results = run_batch(
    llm,
    prompts,
    max_concurrency=3,  # Process 3 at a time
    return_exceptions=True
)
batch_time = time.time() - start

for i, result in enumerate(batch_results, 1):
    if isinstance(result, Exception):
        print(f"{i}. Error: {result}")
    else:
        print(f"{i}. {result.content[:50]}...")

print(f"\nProcessed {len(prompts)} prompts in {batch_time:.2f}s")
print("\n" + "="*80 + "\n")


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│    1 # Batching: Process multiple inputs efficiently                                             │
│ ❱  2 from langchain_core.batch import run_batch                                                  │
│    3                                                                                             │
│    4 prompts = [                                                                                 │
│    5 │   "Summarize: Machine learning is a subset of AI",                                        │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ModuleNotFoundError: No module named 'langchain_core.batch'

## 10. Deployment & Scaling {#deployment}

### API Deployment with FastAPI

FastAPI is ideal for deploying LLM applications due to:
- Async support
- Automatic API documentation
- Type validation
- High performance
